# Chapter 14: Physics-Informed Neural Networks

## Automatic Differentiation

### PyTorch Implementation of Automatic Differentiation

In [ ]:
import torch
import torch.nn as nn

# Define network
net = nn.Sequential(
    nn.Linear(2, 32),
    nn.Tanh(),
    nn.Linear(32, 32),
    nn.Tanh(),
    nn.Linear(32, 1)
)

# Input coordinates (requires gradient tracking)
x = torch.tensor([0.5], requires_grad=True, dtype=torch.float32)
t = torch.tensor([0.2], requires_grad=True, dtype=torch.float32)

# Stack inputs
coords = torch.cat([x, t]).reshape(1, -1)

# Forward pass
u = net(coords)

# First derivatives
u_x = torch.autograd.grad(u, x, create_graph=True)[0]
u_t = torch.autograd.grad(u, t, create_graph=True)[0]

# Second derivatives (retain_graph: u_xx and u_tt share
# the same forward graph)
u_xx = torch.autograd.grad(u_x, x, retain_graph=True)[0]
u_tt = torch.autograd.grad(u_t, t)[0]

print(f"u = {u.item():.6f}")
print(f"u_x = {u_x.item():.6f}")
print(f"u_t = {u_t.item():.6f}")
print(f"u_xx = {u_xx.item():.6f}")
print(f"u_tt = {u_tt.item():.6f}")

## PINN Architecture and Loss Functions

### Sampling Strategies

In [ ]:
import torch
import numpy as np

def uniform_sampling(domain_bounds, n_points):
    """Uniform sampling in [a, b] x [c, d]"""
    x_min, x_max, t_min, t_max = domain_bounds
    x = torch.linspace(x_min, x_max, int(np.sqrt(n_points)))
    t = torch.linspace(t_min, t_max, int(np.sqrt(n_points)))
    X, T = torch.meshgrid(x, t, indexing='ij')
    points = torch.stack([X.flatten(), T.flatten()], dim=1)
    return points

def latin_hypercube_sampling(domain_bounds, n_points):
    """Latin hypercube sampling for high-dimensional efficiency"""
    x_min, x_max, t_min, t_max = domain_bounds
    n_dim = 2
    samples = np.random.rand(n_points, n_dim)
    for i in range(n_dim):
        bounds = [x_min, x_max] if i == 0 else [t_min, t_max]
        samples[:, i] = bounds[0] + samples[:, i] * (bounds[1] - bounds[0])
    return torch.tensor(samples, dtype=torch.float32)

def adaptive_sampling(candidate_points, residuals, n_points):
    """Adaptive refinement: sample more where residuals are high"""
    # Normalize residuals
    res_norm = residuals / residuals.max()
    # Probability proportional to residuals
    probs = res_norm / res_norm.sum()
    # Sample indices with replacement, weighted by residuals
    indices = np.random.choice(len(residuals), size=n_points, p=probs)
    # Perturb the selected candidate points with small noise
    points = candidate_points[indices] + 0.01 * np.random.randn(n_points, 2)
    return torch.tensor(points, dtype=torch.float32)

### Fourier Feature Embeddings

In [ ]:
class FourierFeatureNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, freq_scale=1.0):
        super().__init__()
        self.input_dim = input_dim
        self.freq_scale = freq_scale
        # Learnable frequency matrix
        self.B = nn.Parameter(torch.randn(input_dim, input_dim // 2) * freq_scale)

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        # Fourier features: [sin(2*pi*B*x), cos(2*pi*B*x)]
        z = 2 * torch.pi * x @ self.B
        z_sin = torch.sin(z)
        z_cos = torch.cos(z)
        z_concat = torch.cat([z_sin, z_cos], dim=1)
        return self.net(z_concat)

### PINN Class Implementation

In [ ]:
class PINN(nn.Module):
    def __init__(self, layers, lb, ub, device='cpu'):
        super().__init__()
        self.device = device
        self.lb = torch.tensor(lb, dtype=torch.float32, device=device)
        self.ub = torch.tensor(ub, dtype=torch.float32, device=device)

        # Build network
        self.layers = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.layers.append(nn.Linear(layers[i], layers[i+1]))

    def forward(self, x):
        # Normalize inputs to [-1, 1]
        x_norm = 2 * (x - self.lb) / (self.ub - self.lb) - 1

        for layer in self.layers[:-1]:
            x_norm = torch.tanh(layer(x_norm))

        x_norm = self.layers[-1](x_norm)
        return x_norm

    def compute_pde_residual(self, x, t, pde_fn):
        """Compute residual of a PDE"""
        x.requires_grad_(True)
        t.requires_grad_(True)

        coords = torch.cat([x, t], dim=1)
        u = self.forward(coords)

        u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
        u_t = torch.autograd.grad(u.sum(), t, create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]

        residual = pde_fn(u, u_x, u_t, u_xx)
        return residual

## PINNs for Forward Problems

### Example 1: One-Dimensional Heat Equation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

class HeatEquationPINN(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, t):
        xt = torch.cat([x, t], dim=1)
        return self.net(xt)

    def pde_residual(self, x, t, alpha=0.1):
        x.requires_grad_(True)
        t.requires_grad_(True)

        u = self.forward(x, t)

        u_t = torch.autograd.grad(u.sum(), t, create_graph=True)[0]
        u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]

        residual = u_t - alpha * u_xx
        return residual, u, u_x, u_t, u_xx

# Generate training data
n_residual = 1000
n_boundary = 100
n_initial = 100

x_residual = torch.rand(n_residual, 1)
t_residual = torch.rand(n_residual, 1)

x_boundary = torch.cat([torch.zeros(n_boundary // 2, 1), torch.ones(n_boundary // 2, 1)])
t_boundary = torch.rand(n_boundary, 1)

x_initial = torch.rand(n_initial, 1)
t_initial = torch.zeros(n_initial, 1)

u_initial = torch.sin(np.pi * x_initial)

# Train PINN
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HeatEquationPINN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=0.9)

alpha = 0.1
epochs = 5000

for epoch in range(epochs):
    optimizer.zero_grad()

    # PDE residual loss
    x_r = x_residual.to(device)
    t_r = t_residual.to(device)
    residual, _, _, _, _ = model.pde_residual(x_r, t_r, alpha)
    loss_pde = torch.mean(residual ** 2)

    # Boundary condition loss
    x_b = x_boundary.to(device)
    t_b = t_boundary.to(device)
    u_b = model(x_b, t_b)
    loss_bc = torch.mean(u_b ** 2)

    # Initial condition loss
    x_i = x_initial.to(device)
    t_i = t_initial.to(device)
    u_i = model(x_i, t_i)
    loss_ic = torch.mean((u_i - u_initial.to(device)) ** 2)

    loss = loss_pde + loss_bc + loss_ic

    loss.backward()
    optimizer.step()
    scheduler.step()

    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.6f}")

# Evaluate solution
x_test = torch.linspace(0, 1, 100).reshape(-1, 1)
t_test = torch.linspace(0, 1, 100).reshape(-1, 1)
X_test, T_test = torch.meshgrid(x_test.squeeze(), t_test.squeeze(), indexing='ij')
x_flat = X_test.reshape(-1, 1)
t_flat = T_test.reshape(-1, 1)

u_pred = model(x_flat.to(device), t_flat.to(device)).detach().cpu().numpy()
u_exact = np.sin(np.pi * x_flat.numpy()) * np.exp(-alpha * np.pi**2 * t_flat.numpy())

error = np.abs(u_pred - u_exact)
print(f"Mean Absolute Error: {np.mean(error):.6e}")
print(f"Max Absolute Error: {np.max(error):.6e}")

### Example 2: Two-Dimensional Poisson Equation on Irregular Domain

In [ ]:
class PoissonPINN(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, y):
        xy = torch.cat([x, y], dim=1)
        return self.net(xy)

    def laplacian(self, x, y):
        x.requires_grad_(True)
        y.requires_grad_(True)
        u = self.forward(x, y)
        u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
        u_y = torch.autograd.grad(u.sum(), y, create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]
        u_yy = torch.autograd.grad(u_y.sum(), y, create_graph=True)[0]
        return u_xx + u_yy

# Sample points in disk
def sample_disk(n_interior, n_boundary, radius=0.5):
    # Interior: rejection sampling
    interior_points = []
    while len(interior_points) < n_interior:
        x = torch.rand(1000) * 2 * radius - radius
        y = torch.rand(1000) * 2 * radius - radius
        r = torch.sqrt(x**2 + y**2)
        mask = r < radius
        interior_points.append(torch.stack([x[mask], y[mask]], dim=1))
    interior = torch.cat(interior_points)[:n_interior]

    # Boundary: circle
    theta = torch.linspace(0, 2*np.pi, n_boundary)
    x_boundary = radius * torch.cos(theta).reshape(-1, 1)
    y_boundary = radius * torch.sin(theta).reshape(-1, 1)

    return interior, torch.cat([x_boundary, y_boundary], dim=1)

# Training
model = PoissonPINN()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

xy_interior, xy_boundary = sample_disk(500, 100)

epochs = 3000
for epoch in range(epochs):
    optimizer.zero_grad()

    # PDE: Laplacian = -2
    laplacian = model.laplacian(xy_interior[:, 0:1].clone(),
                                xy_interior[:, 1:2].clone())
    loss_pde = torch.mean((laplacian + 2)**2)

    # BC: u = 0
    u_bc = model(xy_boundary[:, 0:1], xy_boundary[:, 1:2])
    loss_bc = torch.mean(u_bc**2)

    loss = loss_pde + loss_bc
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.6e}")

### Example 3: Burgers' Equation (Nonlinear PDE)

In [ ]:
class BurgersPINN(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, t):
        xt = torch.cat([x, t], dim=1)
        return self.net(xt)

    def burgers_residual(self, x, t, nu=0.01/np.pi):
        x.requires_grad_(True)
        t.requires_grad_(True)

        u = self.forward(x, t)
        u_t = torch.autograd.grad(u.sum(), t, create_graph=True)[0]
        u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]

        residual = u_t + u * u_x - nu * u_xx
        return residual

# Generate data
n_residual = 2000
n_initial = 100
n_boundary = 100

x_residual = 2 * torch.rand(n_residual, 1) - 1
t_residual = torch.rand(n_residual, 1)

x_initial = 2 * torch.rand(n_initial, 1) - 1
t_initial = torch.zeros(n_initial, 1)
u_initial = -torch.sin(np.pi * x_initial)

# Boundary points at x = -1 and x = 1
x_boundary = torch.cat([-torch.ones(n_boundary // 2, 1),
                        torch.ones(n_boundary // 2, 1)])
t_boundary = torch.rand(n_boundary, 1)

# Train
model = BurgersPINN()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 5000
nu = 0.01 / np.pi

for epoch in range(epochs):
    optimizer.zero_grad()

    # PDE residual
    x_r = x_residual.clone()
    t_r = t_residual.clone()
    residual = model.burgers_residual(x_r, t_r, nu)
    loss_residual = torch.mean(residual**2)

    # Initial condition
    x_i = x_initial.clone()
    t_i = t_initial.clone()
    u_i = model(x_i, t_i)
    loss_ic = torch.mean((u_i - u_initial)**2)

    # Boundary condition: u(-1, t) = u(1, t) = 0
    u_b = model(x_boundary, t_boundary)
    loss_bc = torch.mean(u_b**2)

    loss = loss_residual + loss_ic + loss_bc
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.6e}")

### Example 4: Simplified 2D Navier-Stokes (Lid-Driven Cavity)

In [ ]:
class NavierStokesPINN(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        # Network for velocity and pressure
        self.net_u = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        self.net_v = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        self.net_p = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, y):
        xy = torch.cat([x, y], dim=1)
        u = self.net_u(xy)
        v = self.net_v(xy)
        p = self.net_p(xy)
        return u, v, p

    def navier_stokes_residuals(self, x, y, rho=1.0, mu=0.01):
        x.requires_grad_(True)
        y.requires_grad_(True)

        u, v, p = self.forward(x, y)

        # Derivatives
        u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
        u_y = torch.autograd.grad(u.sum(), y, create_graph=True)[0]
        v_x = torch.autograd.grad(v.sum(), x, create_graph=True)[0]
        v_y = torch.autograd.grad(v.sum(), y, create_graph=True)[0]
        p_x = torch.autograd.grad(p.sum(), x, create_graph=True)[0]
        p_y = torch.autograd.grad(p.sum(), y, create_graph=True)[0]

        u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]
        u_yy = torch.autograd.grad(u_y.sum(), y, create_graph=True)[0]
        v_xx = torch.autograd.grad(v_x.sum(), x, create_graph=True)[0]
        v_yy = torch.autograd.grad(v_y.sum(), y, create_graph=True)[0]

        # Continuity
        cont = u_x + v_y

        # Momentum (simplified, no time dependence)
        mom_u = rho * (u * u_x + v * u_y) + p_x - mu * (u_xx + u_yy)
        mom_v = rho * (u * v_x + v * v_y) + p_y - mu * (v_xx + v_yy)

        return cont, mom_u, mom_v

## PINNs for Inverse Problems

### Parameter Identification

In [ ]:
class InversePINN(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        # Learnable parameter, deliberately initialized away
        # from the true value alpha = 0.1
        self.log_alpha = nn.Parameter(torch.tensor(np.log(0.5)))

    @property
    def alpha(self):
        return torch.exp(self.log_alpha)

    def forward(self, x, t):
        xt = torch.cat([x, t], dim=1)
        return self.net(xt)

    def pde_residual(self, x, t):
        x.requires_grad_(True)
        t.requires_grad_(True)

        u = self.forward(x, t)
        u_t = torch.autograd.grad(u.sum(), t, create_graph=True)[0]
        u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]

        residual = u_t - self.alpha * u_xx
        return residual

# Generate synthetic data (true alpha = 0.1)
true_alpha = 0.1
x_data = torch.rand(50, 1)
t_data = torch.rand(50, 1)
u_exact = torch.sin(np.pi * x_data) * torch.exp(-true_alpha * np.pi**2 * t_data)
# Add noise
u_data = u_exact + 0.01 * torch.randn_like(u_exact)

# Collocation points
x_residual = torch.rand(500, 1)
t_residual = torch.rand(500, 1)

# Train
model = InversePINN()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 2000
for epoch in range(epochs):
    optimizer.zero_grad()

    # PDE residual
    residual = model.pde_residual(x_residual, t_residual)
    loss_pde = torch.mean(residual**2)

    # Data mismatch (weighted up: the data carry the
    # information about alpha)
    u_pred = model(x_data, t_data)
    loss_data = torch.mean((u_pred - u_data)**2)

    loss = loss_pde + 10.0 * loss_data
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.6e}, alpha: {model.alpha.item():.6f}")

print(f"Estimated alpha: {model.alpha.item():.6f} (true: {true_alpha})")

### Wave Speed Identification

In [ ]:
class WaveInversePINN(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        self.log_c = nn.Parameter(torch.tensor(np.log(1.0)))

    @property
    def c(self):
        return torch.exp(self.log_c)

    def forward(self, x, t):
        xt = torch.cat([x, t], dim=1)
        return self.net(xt)

    def wave_residual(self, x, t):
        x.requires_grad_(True)
        t.requires_grad_(True)

        u = self.forward(x, t)
        u_tt = torch.autograd.grad(torch.autograd.grad(u.sum(), t, create_graph=True)[0].sum(), t, create_graph=True)[0]
        u_xx = torch.autograd.grad(torch.autograd.grad(u.sum(), x, create_graph=True)[0].sum(), x, create_graph=True)[0]

        residual = u_tt - self.c**2 * u_xx
        return residual

### Bayesian PINNs and Uncertainty Quantification

In [ ]:
class BayesianPINN(nn.Module):
    def __init__(self, hidden_dim=32, dropout_rate=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, t, mc_samples=1):
        xt = torch.cat([x, t], dim=1)
        self.train()  # Enable dropout even at test time
        outputs = [self.net(xt) for _ in range(mc_samples)]
        self.eval()
        return torch.cat(outputs, dim=1)  # Shape: (N, mc_samples)

    def predict_with_uncertainty(self, x, t, mc_samples=100):
        preds = self.forward(x, t, mc_samples)
        mean = preds.mean(dim=1)
        std = preds.std(dim=1)
        return mean, std

## Advanced PINN Techniques

### Time-Stepping PINNs

In [ ]:
def train_time_stepping_pinn(model, optimizer, domain_bounds, time_intervals,
                             get_previous_solution,
                             n_residual_per_interval=500):
    """Train PINN with sequential time stepping.

    get_previous_solution(x, t): callable returning the solution
    learned on the previous interval, evaluated at (x, t).
    """
    x_min, x_max = domain_bounds

    for interval_idx, (t_start, t_end) in enumerate(time_intervals):
        print(f"Training interval {interval_idx+1}/{len(time_intervals)}: t in [{t_start}, {t_end}]")

        # Generate residual points in this interval
        x_residual = torch.rand(n_residual_per_interval, 1) * (x_max - x_min) + x_min
        t_residual = torch.rand(n_residual_per_interval, 1) * (t_end - t_start) + t_start

        # If not first interval, enforce continuity with previous solution
        if interval_idx > 0:
            x_ic = torch.linspace(x_min, x_max, 100).reshape(-1, 1)
            t_ic = torch.full_like(x_ic, t_start)
            u_prev = get_previous_solution(x_ic, t_ic).detach()

        # Train on this interval
        for epoch in range(200):
            optimizer.zero_grad()

            residual = model.pde_residual(x_residual, t_residual)
            loss = torch.mean(residual**2)

            if interval_idx > 0:
                u_curr = model(x_ic, t_ic)
                loss += 0.1 * torch.mean((u_curr - u_prev)**2)

            loss.backward()
            optimizer.step()

### Transfer Learning for Similar PDEs

In [ ]:
def train_pinn(model, alpha, epochs, optimizer=None, n_points=1000):
    """Minimal residual-only training loop for illustration."""
    if optimizer is None:
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
    x = torch.rand(n_points, 1)
    t = torch.rand(n_points, 1)
    for epoch in range(epochs):
        optimizer.zero_grad()
        residual, _, _, _, _ = model.pde_residual(x, t, alpha)
        loss = torch.mean(residual**2)
        loss.backward()
        optimizer.step()

# Train on source PDE: alpha = 0.1
model_source = HeatEquationPINN()
train_pinn(model_source, alpha=0.1, epochs=1000)

# Transfer to target PDE: alpha = 0.05
model_target = HeatEquationPINN()
model_target.load_state_dict(model_source.state_dict())

# Fine-tune on target
optimizer = optim.Adam(model_target.parameters(), lr=1e-4)  # Lower LR
train_pinn(model_target, alpha=0.05, epochs=500, optimizer=optimizer)

### Residual-Based Adaptive Refinement (RAR)

In [ ]:
def train_with_rar(model, optimizer, domain, n_initial=500, refinement_epochs=5):
    """Train with residual-based adaptive refinement"""

    # Initial uniform sampling
    x_residual = torch.rand(n_initial, 1) * (domain[1] - domain[0]) + domain[0]
    t_residual = torch.rand(n_initial, 1)

    for refine_step in range(refinement_epochs):
        print(f"Refinement step {refine_step+1}/{refinement_epochs}")

        for epoch in range(100):
            optimizer.zero_grad()

            residual = model.pde_residual(x_residual, t_residual)
            loss = torch.mean(residual**2)
            loss.backward()
            optimizer.step()

        # Residual magnitudes guide the resampling below
        residual_magnitudes = residual.detach().abs()

        # Adaptive sampling: sample more where residuals are large
        residual_norm = residual_magnitudes / residual_magnitudes.max()
        probs = residual_norm / residual_norm.sum()

        # Resample: keep 80% of old points, add 20% new based on residuals
        n_keep = int(0.8 * n_initial)
        n_new = n_initial - n_keep

        indices = torch.multinomial(probs.squeeze(), n_new, replacement=True)
        x_new = x_residual[indices] + 0.02 * torch.randn_like(x_residual[indices])
        t_new = t_residual[indices] + 0.02 * torch.randn_like(t_residual[indices])

        # detach: the resampled points must not carry autograd
        # history into the next training round
        keep_indices = torch.randperm(len(x_residual))[:n_keep]
        x_residual = torch.cat([x_residual[keep_indices], x_new]).detach()
        t_residual = torch.cat([t_residual[keep_indices], t_new]).detach()

## Applications in Engineering

### Fluid Mechanics: Flow Reconstruction

In [ ]:
class FlowReconstructionPINN(nn.Module):
    """Reconstruct flow from sparse velocity measurements"""
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net_u = nn.Sequential(nn.Linear(3, hidden_dim), nn.Tanh(),
                                    nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
                                    nn.Linear(hidden_dim, 1))
        self.net_v = nn.Sequential(nn.Linear(3, hidden_dim), nn.Tanh(),
                                    nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
                                    nn.Linear(hidden_dim, 1))
        self.net_p = nn.Sequential(nn.Linear(3, hidden_dim), nn.Tanh(),
                                    nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
                                    nn.Linear(hidden_dim, 1))

    def forward(self, x, y, t):
        xyt = torch.cat([x, y, t], dim=1)
        u = self.net_u(xyt)
        v = self.net_v(xyt)
        p = self.net_p(xyt)
        return u, v, p

## Uncertainty Quantification with PINNs

### Approximations to the Posterior

#### Monte Carlo Dropout

In [ ]:
class BayesianPINN(nn.Module):
    def __init__(self, layers, dropout_rate=0.1):
        super().__init__()
        modules = []
        for i in range(len(layers) - 1):
            modules.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers) - 2:  # hidden layers only
                modules.append(nn.Tanh())
                modules.append(nn.Dropout(dropout_rate))
        self.net = nn.Sequential(*modules)

    def forward(self, x, dropout_enabled=False):
        if dropout_enabled:
            self.train()  # Enable dropout
        else:
            self.eval()
        return self.net(x)

    def predict_with_uncertainty(self, x, t, num_samples=100):
        """Perform MC dropout to estimate uncertainty."""
        xt = torch.cat([x, t], dim=1)
        predictions = []

        for _ in range(num_samples):
            pred = self.forward(xt, dropout_enabled=True)
            predictions.append(pred.detach())

        predictions = torch.cat(predictions, dim=1)  # (N, num_samples)
        mean = predictions.mean(dim=1, keepdim=True)
        std = predictions.std(dim=1, keepdim=True)

        return mean, std

#### Ensemble Methods

In [ ]:
class EnsemblePINN:
    def __init__(self, num_models=10, layers=[2, 64, 64, 64, 1]):
        self.models = [BayesianPINN(layers) for _ in range(num_models)]
        self.num_models = num_models

    def train_ensemble(self, x_residual, t_residual, x_data, t_data, u_data,
                      epochs=1000, alpha=0.1):
        """Train ensemble of independent PINNs."""
        for model_idx, model in enumerate(self.models):
            print(f"Training model {model_idx+1}/{self.num_models}")
            optimizer = optim.Adam(model.parameters(), lr=1e-3)

            for epoch in range(epochs):
                optimizer.zero_grad()

                # PDE residual: enable gradients *before*
                # concatenating, so the graph reaches x and t
                x_residual.requires_grad_(True)
                t_residual.requires_grad_(True)
                xt_r = torch.cat([x_residual, t_residual], dim=1)

                u = model(xt_r)
                u_t = torch.autograd.grad(u.sum(), t_residual,
                                         create_graph=True)[0]
                u_x = torch.autograd.grad(u.sum(), x_residual,
                                         create_graph=True)[0]
                u_xx = torch.autograd.grad(u_x.sum(), x_residual,
                                          create_graph=True)[0]

                loss_pde = torch.mean((u_t - alpha * u_xx)**2)

                # Data mismatch
                xt_data = torch.cat([x_data, t_data], dim=1)
                u_pred = model(xt_data)
                loss_data = torch.mean((u_pred - u_data)**2)

                loss = loss_pde + 0.01 * loss_data
                loss.backward()
                optimizer.step()

                x_residual = x_residual.detach()
                t_residual = t_residual.detach()

                if (epoch + 1) % 200 == 0:
                    print(f"  Model {model_idx+1}, Epoch {epoch+1}/{epochs}: "
                          f"Loss={loss.item():.6e}")

    def predict_with_uncertainty(self, x, t):
        """Ensemble prediction with uncertainty."""
        xt = torch.cat([x, t], dim=1)
        predictions = []

        for model in self.models:
            model.eval()
            with torch.no_grad():
                pred = model(xt)
            predictions.append(pred)

        predictions = torch.cat(predictions, dim=1)  # (N, num_models)
        mean = predictions.mean(dim=1, keepdim=True)
        std = predictions.std(dim=1, keepdim=True)

        return mean, std

## PINNs for Coupled Multi-Physics Systems

### PyTorch Implementation: Coupled Burgers-Advection

In [ ]:
class CoupledBurgersPINN(nn.Module):
    """PINN for coupled Burgers (momentum) and scalar transport."""
    def __init__(self, hidden_dim=64):
        super().__init__()
        # Shared feature layers
        self.shared = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()
        )
        # Task-specific heads
        self.head_u = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        self.head_c = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, t):
        xt = torch.cat([x, t], dim=1)
        features = self.shared(xt)
        u = self.head_u(features)
        c = self.head_c(features)
        return u, c

    def residuals(self, x, t, nu=0.01/np.pi, D=0.01/np.pi):
        """Compute residuals for both equations."""
        x.requires_grad_(True)
        t.requires_grad_(True)

        u, c = self.forward(x, t)

        # Burgers equation derivatives
        u_t = torch.autograd.grad(u.sum(), t, create_graph=True)[0]
        u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]
        burgers_res = u_t + u * u_x - nu * u_xx

        # Transport equation derivatives
        c_t = torch.autograd.grad(c.sum(), t, create_graph=True)[0]
        c_x = torch.autograd.grad(c.sum(), x, create_graph=True)[0]
        c_xx = torch.autograd.grad(c_x.sum(), x, create_graph=True)[0]
        transport_res = c_t + u.detach() * c_x - D * c_xx

        return burgers_res, transport_res

## Transfer Learning and Meta-Learning for PINNs

### Pre-training Strategy

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

class HeatEquationPINN(nn.Module):
    """PINN for heat equation with parameter."""

    def __init__(self, hidden_dim=64, n_layers=4):
        super().__init__()
        layers = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        xt = torch.cat([x, t], dim=1)
        return self.net(xt)

    def compute_residual(self, x, t, alpha):
        """Compute PDE residual: u_t = alpha*u_xx"""
        u = self(x, t)

        # Derivatives
        u_t = torch.autograd.grad(u.sum(), t, create_graph=True)[0]
        u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]

        residual = u_t - alpha * u_xx
        return residual

def pretrain_on_parameter_family():
    """Pre-train PINN on heat equation with varying diffusion."""
    torch.manual_seed(42)

    model = HeatEquationPINN(hidden_dim=64, n_layers=4)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Training data for multiple alpha values
    alphas = [0.001, 0.01, 0.05, 0.1]
    n_samples_per_alpha = 2500

    pretraining_losses = []

    for epoch in range(300):
        total_loss = 0.0
        optimizer.zero_grad()  # once per epoch: gradients
                               # accumulate over all alphas

        for alpha in alphas:
            # Residual points
            x_r = (torch.rand(n_samples_per_alpha, 1) * 2 - 1)\
                  .requires_grad_(True)
            t_r = torch.rand(n_samples_per_alpha, 1).requires_grad_(True)

            # IC: u(x, 0) = sin(pi*x)
            x_i = torch.linspace(-1, 1, 500).reshape(-1, 1)
            t_i = torch.zeros(500, 1)
            u_i = torch.sin(np.pi * x_i)

            # BC: u(-1, t) = u(1, t) = 0
            t_b = torch.rand(500, 1)
            x_b_left = -torch.ones(250, 1)
            x_b_right = torch.ones(250, 1)
            u_b = torch.zeros(500, 1)

            optimizer.zero_grad()

            # Residual loss
            residual = model.compute_residual(x_r, t_r, alpha)
            loss_pde = torch.mean(residual**2)

            # IC loss
            u_pred_i = model(x_i, t_i)
            loss_ic = torch.mean((u_pred_i - u_i)**2)

            # BC loss
            u_pred_bl = model(x_b_left, t_b[:250])
            u_pred_br = model(x_b_right, t_b[250:])
            loss_bc = torch.mean((u_pred_bl)**2) + torch.mean((u_pred_br)**2)

            loss = loss_pde + loss_ic + loss_bc

            loss.backward()
            total_loss += loss.item()

        optimizer.step()
        pretraining_losses.append(total_loss)

        if (epoch + 1) % 50 == 0:
            print(f"Pretraining Epoch {epoch+1}, Loss: {total_loss:.6e}")

    return model, pretraining_losses

def finetune_on_target(model, target_alpha, finetune_epochs=200):
    """Fine-tune pre-trained model on specific alpha."""
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    finetuning_losses = []

    for epoch in range(finetune_epochs):
        # Generate fresh data for target alpha
        x_r = (torch.rand(2500, 1) * 2 - 1).requires_grad_(True)
        t_r = torch.rand(2500, 1).requires_grad_(True)

        # IC: u(x, 0) = sin(pi*x)
        x_i = torch.linspace(-1, 1, 500).reshape(-1, 1)
        t_i = torch.zeros(500, 1)
        u_i = torch.sin(np.pi * x_i)

        # BC
        t_b = torch.rand(500, 1)
        x_b_left = -torch.ones(250, 1)
        x_b_right = torch.ones(250, 1)

        optimizer.zero_grad()

        residual = model.compute_residual(x_r, t_r, target_alpha)
        loss_pde = torch.mean(residual**2)

        u_pred_i = model(x_i, t_i)
        loss_ic = torch.mean((u_pred_i - u_i)**2)

        u_pred_bl = model(x_b_left, t_b[:250])
        u_pred_br = model(x_b_right, t_b[250:])
        loss_bc = torch.mean((u_pred_bl)**2) + torch.mean((u_pred_br)**2)

        loss = loss_pde + loss_ic + loss_bc

        loss.backward()
        optimizer.step()

        finetuning_losses.append(loss.item())

        if (epoch + 1) % 50 == 0:
            print(f"Finetuning Epoch {epoch+1}, Loss: {loss.item():.6e}")

    return finetuning_losses

# Pre-train and fine-tune
print("Starting pre-training on parameter family...")
model_pretrained, pretrain_losses = pretrain_on_parameter_family()

print("\nStarting fine-tuning on target alpha = 0.02...")
target_alpha = 0.02
finetune_losses = finetune_on_target(model_pretrained, target_alpha)

# Plot convergence
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(pretrain_losses, 'b-', linewidth=2, label='Pretraining')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Total loss (dimensionless)', fontsize=11)
axes[0].set_title('Pre-training on Parameter Family', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=10)

axes[1].semilogy(finetune_losses, 'r-', linewidth=2, label='Fine-tuning')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Total loss (dimensionless)', fontsize=11)
axes[1].set_title(f'Fine-tuning for target alpha={target_alpha}', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('figures/chapter_14_transfer_learning.png', dpi=200)
plt.show()

## Active Learning and Adaptive Training for PINNs

### Residual-Based Adaptive Refinement

In [ ]:
def adaptive_collocation_sampling(model, domain_bounds, n_initial=1000,
                                 n_iterations=5, refinement_fraction=0.5):
    """
    Adaptively refine collocation point distribution based on residuals.

    model: PINN that computes residuals
    domain_bounds: (x_min, x_max, t_min, t_max)
    """
    x_min, x_max, t_min, t_max = domain_bounds

    # Start with uniform sampling
    x_initial = torch.rand(n_initial, 1) * (x_max - x_min) + x_min
    t_initial = torch.rand(n_initial, 1) * (t_max - t_min) + t_min

    x_colloc = x_initial.clone()
    t_colloc = t_initial.clone()

    residual_history = []

    for iteration in range(n_iterations):
        print(f"Adaptive iteration {iteration+1}/{n_iterations}")

        # Compute residuals at current points
        x_colloc.requires_grad_(True)
        t_colloc.requires_grad_(True)

        residuals = model.compute_residual(x_colloc, t_colloc)
        residual_magnitudes = torch.abs(residuals.detach())

        residual_history.append(residual_magnitudes.mean().item())

        # Create probability distribution from residuals
        residual_prob = residual_magnitudes / residual_magnitudes.sum()

        # Sample new points from high-residual regions
        n_new = int(refinement_fraction * len(x_colloc))
        indices = torch.multinomial(residual_prob.squeeze(), n_new,
                                   replacement=True)

        # Perturb selected points
        noise_scale = (x_max - x_min) * 0.02
        x_new = (x_colloc[indices] +
                torch.randn_like(x_colloc[indices]) * noise_scale)
        t_new = (t_colloc[indices] +
                torch.randn_like(t_colloc[indices]) * noise_scale)

        # Keep old points for stability
        keep_fraction = 0.7
        keep_idx = torch.randperm(len(x_colloc))[:int(keep_fraction * len(x_colloc))]

        # Combine old and new; detach so the new point set is a
        # fresh leaf tensor without autograd history
        x_colloc = torch.cat([x_colloc[keep_idx], x_new]).detach()
        t_colloc = torch.cat([t_colloc[keep_idx], t_new]).detach()

        print(f"  Collocation points: {len(x_colloc)}, Mean residual: {residual_magnitudes.mean():.6e}")

    return x_colloc, t_colloc, residual_history

# Example: Adaptive sampling for Burgers' equation
# (Would use actual PINN model in practice)

### Error-Driven Sampling

In [ ]:
def error_driven_sampling(model, loss_history_per_point, n_new=500):
    """
    Sample new collocation points from regions where training loss is high.
    """
    # Normalize loss history
    loss_weights = loss_history_per_point / loss_history_per_point.sum()

    # Sample indices according to loss distribution
    new_indices = torch.multinomial(loss_weights, n_new, replacement=True)

    return new_indices

### Curriculum Learning for PDEs

In [ ]:
def curriculum_learning_pinn(base_model, pde_family, difficulty_levels):
    """
    Curriculum learning: train on easy problems first, then harder ones.

    pde_family: list of PDEs ordered by difficulty
    difficulty_levels: corresponding difficulty scores
    """
    current_model = base_model

    for difficulty, pde in zip(difficulty_levels, pde_family):
        print(f"Training on difficulty level {difficulty:.2f}...")

        # Reduce learning rate for harder problems
        lr = 0.001 / (1 + 2*difficulty)
        optimizer = optim.Adam(current_model.parameters(), lr=lr)

        # Train for fewer epochs on harder problems
        epochs = int(500 / (1 + difficulty))

        for epoch in range(epochs):
            # Training loop...
            pass

        print(f"  Completed difficulty {difficulty:.2f}")

    return current_model

## PINNs for Stochastic PDEs

### Polynomial Chaos Expansion with PINNs

In [ ]:
class StochasticPINN(nn.Module):
    """PINN for stochastic PDEs using polynomial chaos."""

    def __init__(self, n_chaos_modes=10, hidden_dim=64):
        super().__init__()
        self.n_modes = n_chaos_modes

        # Separate networks for each chaos mode coefficient
        self.mode_networks = nn.ModuleList([
            nn.Sequential(
                nn.Linear(2, hidden_dim),
                nn.Tanh(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.Tanh(),
                nn.Linear(hidden_dim, 1)
            )
            for _ in range(n_chaos_modes)
        ])

    def forward(self, x, t, chaos_samples):
        """
        chaos_samples: (batch, n_random_vars) - samples of random inputs
        """
        batch_size = x.shape[0]
        u_mean = torch.zeros(batch_size, 1)

        # Hermite polynomials (for Gaussian random variables)
        hermite_vals = [
            torch.ones_like(chaos_samples[:, 0:1]),  # H_0 = 1
            chaos_samples[:, 0:1],  # H_1 = xi
            chaos_samples[:, 0:1]**2 - 1,  # H_2 = xi^2 - 1
        ]

        for mode_idx, net in enumerate(self.mode_networks):
            if mode_idx < len(hermite_vals):
                c_k = net(torch.cat([x, t], dim=1))
                u_mean = u_mean + c_k * hermite_vals[mode_idx]

        return u_mean

    def compute_residual(self, x, t, alpha, chaos_samples):
        """Compute PDE residual with stochastic terms."""
        u = self.forward(x, t, chaos_samples)

        u_t = torch.autograd.grad(u.sum(), t, create_graph=True)[0]
        u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]

        # Deterministic part of the operator; randomness
        # enters through the chaos modes in the forward pass
        residual = u_t - alpha * u_xx

        return residual

# Example: Solve stochastic heat equation
model_stoch = StochasticPINN(n_chaos_modes=5, hidden_dim=64)
optimizer_stoch = optim.Adam(model_stoch.parameters(), lr=0.001)

n_samples = 1000
for epoch in range(100):
    x_r = torch.rand(n_samples, 1).requires_grad_(True)
    t_r = torch.rand(n_samples, 1).requires_grad_(True)

    # Sample random variables (uniform, Gaussian, etc.)
    xi = torch.randn(n_samples, 2)

    optimizer_stoch.zero_grad()

    residual = model_stoch.compute_residual(x_r, t_r, alpha=0.01, chaos_samples=xi)
    loss = torch.mean(residual**2)

    loss.backward()
    optimizer_stoch.step()

    if (epoch + 1) % 25 == 0:
        print(f"Stochastic PINN - Epoch {epoch+1}, Loss: {loss.item():.6e}")

## Industrial Case Studies

### Groundwater Flow Modeling

In [ ]:
class GroundwaterFlowPINN(nn.Module):
    """PINN for groundwater flow with unknown heterogeneous conductivity."""

    def __init__(self, hidden_dim=64):
        super().__init__()

        # Network for hydraulic head h(x, y, t)
        self.h_net = nn.Sequential(
            nn.Linear(3, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

        # Network for log-conductivity log(K(x, y))
        self.logK_net = nn.Sequential(
            nn.Linear(2, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward_h(self, x, y, t):
        """Predict hydraulic head."""
        return self.h_net(torch.cat([x, y, t], dim=1))

    def forward_K(self, x, y):
        """Predict conductivity."""
        return torch.exp(self.logK_net(torch.cat([x, y], dim=1)))

    def compute_residual(self, x, y, t, Ss=0.0001):
        """Compute PDE residual: Ss*h_t = div(K*grad(h))"""
        x.requires_grad_(True)
        y.requires_grad_(True)
        t.requires_grad_(True)

        h = self.forward_h(x, y, t)
        K = self.forward_K(x, y)

        h_t = torch.autograd.grad(h.sum(), t, create_graph=True)[0]
        h_x = torch.autograd.grad(h.sum(), x, create_graph=True)[0]
        h_y = torch.autograd.grad(h.sum(), y, create_graph=True)[0]

        K_x = torch.autograd.grad(K.sum(), x, create_graph=True)[0]
        K_y = torch.autograd.grad(K.sum(), y, create_graph=True)[0]

        h_xx = torch.autograd.grad(h_x.sum(), x, create_graph=True)[0]
        h_yy = torch.autograd.grad(h_y.sum(), y, create_graph=True)[0]

        # Divergence: div(K*grad(h)) = K_x*h_x + K*h_xx + K_y*h_y + K*h_yy
        div_flux = K_x * h_x + K * h_xx + K_y * h_y + K * h_yy

        residual = Ss * h_t - div_flux

        return residual

# Training with observations
print("Training Groundwater PINN with inverse problem...")

model_gw = GroundwaterFlowPINN(hidden_dim=64)
optimizer_gw = optim.Adam(model_gw.parameters(), lr=0.001)

# Synthetic observation data (locations and measured heads).
# Coordinates are scaled to [0, 1]: x, y in km, t in years.
# Unnormalized inputs (e.g., metres up to 1000) saturate the
# tanh activations and stall training.
n_obs = 100
x_obs = torch.rand(n_obs, 1)   # Domain [0, 1] km
y_obs = torch.rand(n_obs, 1)
t_obs = torch.rand(n_obs, 1)   # 1 year
# Head in metres: 2 m/km gradient in x, 1 m/km in y, 0.1 m noise
h_obs = 10 + 2.0 * x_obs + 1.0 * y_obs + torch.randn(n_obs, 1) * 0.1

# Collocation points
n_coll = 5000
x_coll = torch.rand(n_coll, 1)
y_coll = torch.rand(n_coll, 1)
t_coll = torch.rand(n_coll, 1)

for epoch in range(500):
    optimizer_gw.zero_grad()

    # PDE residual loss
    residual = model_gw.compute_residual(x_coll, y_coll, t_coll)
    loss_pde = torch.mean(residual**2)

    # Observation loss
    h_pred = model_gw.forward_h(x_obs, y_obs, t_obs)
    loss_obs = torch.mean((h_pred - h_obs)**2)

    # Regularization on conductivity (smoothness)
    K_vals = model_gw.forward_K(x_obs, y_obs)
    loss_reg = torch.mean((K_vals - K_vals.mean())**2)

    total_loss = loss_pde + 10*loss_obs + 0.01*loss_reg

    total_loss.backward()
    optimizer_gw.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}: PDE={loss_pde:.2e}, Obs={loss_obs:.2e}, "
              f"Total={total_loss:.2e}")